In [ ]:
from pathlib import Path

import h5py
import nibabel as nib
import numpy as np


BASE_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol05_Dat_NoL2_GradDel/maps"
)


lcm_fit = nib.load(
    BASE_PATH / "SpecMap_LCMFit.nii.gz"
).get_fdata()

baseline = nib.load(
    BASE_PATH / "SpecMap_LCMBaseline.nii.gz"
).get_fdata()

metabolites = (
    lcm_fit
    - baseline
)

brain_mask = (
    nib.load(
        BASE_PATH / "mask.nii.gz"
    ).get_fdata()
    > 0
)

combined_csi = h5py.File(
    BASE_PATH.parent / "CombinedCSI.mat",
    "r",
)


print("LCModel fit:", lcm_fit.shape)
print("Baseline:", baseline.shape)
print("Metabolites:", metabolites.shape)
print("Brain mask:", brain_mask.shape)

print("\nCombinedCSI keys:")
for key in combined_csi.keys():
    print(" ", key)

In [ ]:
csi = combined_csi["csi"]

print("CSI keys:")
for key in csi.keys():
    print(" ", key)

data = csi["Data"][:]

print()
print("Data shape:", data.shape)
print("Data dtype:", data.dtype)

In [ ]:
data = csi["Data"][:]

data = (
    data["real"]
    + 1j * data["imag"]
)

print(data.shape)
print(data.dtype)

In [ ]:
data = np.transpose(
    data,
    (3, 2, 1, 0),
)

data = np.flip(
    data,
    axis=0,
)

data = np.flip(
    data,
    axis=1,
)

In [ ]:
data.shape

In [ ]:
import matplotlib.pyplot as plt


fid = 5

for z in range(25, 26):

    plt.figure(figsize=(6, 6))

    plt.imshow(
        np.abs(
            data[:, :, z, fid]
        ),
        origin="lower",
        cmap="gray",
    )

    plt.imshow(
        brain_mask[:, :, z],
        origin="lower",
        cmap="Reds",
        alpha=0.5,
    )

    plt.title(f"z = {z}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ----------------------------------------------------------
# Load LCModel input spectrum
# ----------------------------------------------------------

lcm_input = nib.load(
    BASE_PATH / "SpecMap_LCMInput.nii.gz"
).get_fdata()


# ----------------------------------------------------------
# FFT of full spectrum
# ----------------------------------------------------------

full_spectra = np.fft.fft(
    data,
    axis=-1,
)


frequency_hz = np.fft.fftfreq(
    full_spectra.shape[-1],
    d=1 / 2778.0,
)

ppm = (
    4.68
    - frequency_hz / 297.22
)


# ----------------------------------------------------------
# LCModel fit range (same range as SpecMap_LCMInput)
# ----------------------------------------------------------

fit_mask = (
    (ppm >= 1.8)
    & (ppm <= 4.2)
)


# ----------------------------------------------------------
# Normalization factors
# ----------------------------------------------------------

lcm_scale = np.max(
    np.abs(
        lcm_input
    ),
    axis=-1,
)

full_scale = np.max(
    np.abs(
        full_spectra[..., fit_mask]
    ),
    axis=-1,
)


# Avoid division by zero
lcm_scale = np.maximum(
    lcm_scale,
    1e-12,
)

full_scale = np.maximum(
    full_scale,
    1e-12,
)


# ----------------------------------------------------------
# Normalize everything
# ----------------------------------------------------------

lcm_fit_norm = (
    lcm_fit
    / lcm_scale[..., None]
)

baseline_norm = (
    baseline
    / lcm_scale[..., None]
)

metabolites_norm = (
    metabolites
    / lcm_scale[..., None]
)

full_spectra_norm = (
    full_spectra
    / full_scale[..., None]
)


print("Normalization finished.")

In [ ]:
# ----------------------------------------------------------
# Maxima after normalization
# ----------------------------------------------------------

water_mask = (
    (ppm >= 4.3)
    & (ppm <= 5.0)
)

lipid_mask = (
    (ppm >= 0.0)
    & (ppm <= 2.0)
)


water_max = np.max(
    np.abs(
        full_spectra_norm[..., water_mask]
    ),
    axis=-1,
)

lipid_max = np.max(
    np.abs(
        full_spectra_norm[..., lipid_mask]
    ),
    axis=-1,
)

metabolite_max = np.max(
    np.abs(
        metabolites_norm
    ),
    axis=-1,
)


print("Water max:", water_max.shape)
print("Lipid max:", lipid_max.shape)
print("Metabolite max:", metabolite_max.shape)

In [ ]:
valid = (
    brain_mask
    & np.isfinite(water_max)
    & np.isfinite(lipid_max)
    & np.isfinite(metabolite_max)
    & (metabolite_max > 0)
)

water_ratio = (
    water_max[valid]
    / metabolite_max[valid]
)

lipid_ratio = (
    lipid_max[valid]
    / metabolite_max[valid]
)

water_ratio = water_ratio[
    np.isfinite(water_ratio)
    & (water_ratio > 0)
]

lipid_ratio = lipid_ratio[
    np.isfinite(lipid_ratio)
    & (lipid_ratio > 0)
]

print(
    "Water ratio:",
    water_ratio.shape,
)

print(
    "Lipid ratio:",
    lipid_ratio.shape,
)

In [ ]:
water_xlim = np.percentile(
    water_ratio,
    99,
)

lipid_xlim = np.percentile(
    lipid_ratio,
    99,
)


plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(
    water_ratio,
    bins=200,
    density=True,
    range=(0, water_xlim),
)
plt.xlim(0, water_xlim)
plt.title("Water / Metabolites")
plt.xlabel(r"$\max|S_W|/\max|S_M|$")
plt.ylabel("Density")

plt.subplot(1, 2, 2)
plt.hist(
    lipid_ratio,
    bins=200,
    density=True,
    range=(0, lipid_xlim),
)
plt.xlim(0, lipid_xlim)
plt.title("Lipids / Metabolites")
plt.xlabel(r"$\max|S_L|/\max|S_M|$")
plt.ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import norm


def plot_simulation_distribution(
    values,
    ax,
    title,
    iqr_factor,
):
    median = np.median(values)

    iqr = (
        np.percentile(values, 75)
        - np.percentile(values, 25)
    )

    mu = median
    sigma = iqr_factor * iqr

    xmax = np.percentile(
        values,
        99,
    )

    ax.hist(
        values,
        bins=100,
        range=(0, xmax),
        density=True,
        alpha=0.6,
        label="In-vivo",
    )

    x = np.linspace(
        0,
        xmax,
        1000,
    )

    ax.plot(
        x,
        norm.pdf(
            x,
            loc=mu,
            scale=sigma,
        ),
        lw=2,
        label=(
            f"Simulation\n"
            f"$\\mu$={mu:.2f}, "
            f"$\\sigma$={sigma:.2f}"
        ),
    )

    ax.axvline(
        median,
        ls="--",
        lw=1,
        label=f"Median={median:.2f}",
    )

    ax.set_title(title)
    ax.set_xlabel("Ratio")
    ax.set_ylabel("Density")
    ax.legend()


fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

plot_simulation_distribution(
    water_ratio,
    axes[0],
    "Water / Metabolites",
    iqr_factor=2,
)

plot_simulation_distribution(
    lipid_ratio,
    axes[1],
    "Lipids / Metabolites",
    iqr_factor=3,
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def sample_positive_normal(
    mean,
    std,
    size,
    seed=12345,
):
    rng = np.random.default_rng(
        seed
    )

    values = rng.normal(
        loc=mean,
        scale=std,
        size=size,
    )

    invalid = values <= 0

    while np.any(invalid):
        values[invalid] = rng.normal(
            loc=mean,
            scale=std,
            size=np.sum(invalid),
        )

        invalid = values <= 0

    return values


lipid_median = np.median(
    lipid_ratio
)

lipid_iqr = (
    np.percentile(
        lipid_ratio,
        75,
    )
    - np.percentile(
        lipid_ratio,
        25,
    )
)

lipid_sigma = (
    3.0
    * lipid_iqr
)

simulated_lipid_ratio = (
    sample_positive_normal(
        mean=lipid_median,
        std=lipid_sigma,
        size=500_000,
    )
)

xmax = max(
    np.percentile(
        lipid_ratio,
        99,
    ),
    np.percentile(
        simulated_lipid_ratio,
        99,
    ),
)


plt.figure(
    figsize=(7, 5)
)

plt.hist(
    lipid_ratio,
    bins=150,
    range=(0, xmax),
    density=True,
    alpha=0.6,
    label="In-vivo",
)

plt.hist(
    simulated_lipid_ratio,
    bins=150,
    range=(0, xmax),
    density=True,
    histtype="step",
    linewidth=2,
    label=(
        "Positive normal sampler\n"
        f"$\\mu$={lipid_median:.2f}, "
        f"$\\sigma$={lipid_sigma:.2f}"
    ),
)

plt.axvline(
    lipid_median,
    linestyle="--",
    linewidth=1,
    label=(
        f"Configured location="
        f"{lipid_median:.2f}"
    ),
)

plt.xlabel(
    r"$\max|S_L|/\max|S_M|$"
)
plt.ylabel(
    "Density"
)
plt.title(
    "Lipid scaling: in-vivo vs simulation sampler"
)
plt.legend()
plt.tight_layout()
plt.show()


print(
    "Configured location:",
    lipid_median,
)

print(
    "Configured std:",
    lipid_sigma,
)

print(
    "Sampled mean:",
    simulated_lipid_ratio.mean(),
)

print(
    "Sampled median:",
    np.median(
        simulated_lipid_ratio
    ),
)

print(
    "Sampled IQR:",
    np.percentile(
        simulated_lipid_ratio,
        75,
    )
    - np.percentile(
        simulated_lipid_ratio,
        25,
    ),
)